In [ ]:
from google.colab import files,output
output.enable_custom_widget_manager()

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

uploaded=files.upload()
file_name=next(iter(uploaded))
df=pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Number of records : ",len(df))
df.head()

Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k (1).csv
Dataset loaded successfully!
Number of records :  12500


,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0120001,2026-06-01,PL05,MC024,OP0191,Evening,PMP-210,Pumps,293,292,10,44.0,120.42,70.59,2.21,Pass,Yes,Anita
1,PR0120002,2026-09-22,PL01,MC001,OP0461,Night,MTR-100,Motors,1009,897,25,59.0,317.23,81.74,3.03,Pass,No,Anita
2,PR0120003,2026-05-24,PL03,MC012,OP0451,Evening,MTR-100,Motors,960,983,34,0.0,350.82,67.29,2.01,Pass,No,Deepa
3,PR0120004,2026-09-12,PL04,MC016,OP0237,Night,CTL-130,Controllers,310,320,15,14.0,142.34,59.70,3.48,Pass,No,Anita
4,PR0120005,2026-02-04,PL04,MC018,OP0072,Morning,VLV-220,Valves,957,979,22,14.0,296.70,62.52,2.33,Pass,No,Harini


In [ ]:
# Function to create dropdown options
def create_options(column, all_label):
    values = sorted(df[column].dropna().astype(str).unique())

    return [(all_label, "__ALL__")] + [
        (value, value) for value in values
    ]


In [ ]:
# 1. Plant dropdown
plant_dropdown = widgets.Dropdown(
    options=create_options("plant_id", "All Plants"),
    description="Plant:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)

In [ ]:
# 2. Product family dropdown
product_dropdown = widgets.Dropdown(
    options=create_options("product_family", "All Products"),
    description="Product Family:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)

In [ ]:
# 3. Quality status dropdown
quality_dropdown = widgets.Dropdown(
    options=create_options("quality_status", "All Statuses"),
    description="Quality:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)

In [ ]:


# 4. Slider for units produced
units = pd.to_numeric(df["units_produced"], errors="coerce").dropna()

units_slider = widgets.IntRangeSlider(
    value=(int(units.min()), int(units.max())),
    min=int(units.min()),
    max=int(units.max()),
    step=1,
    description="Units Produced:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%")
)

In [ ]:
from IPython.display import display
from google.colab import output

# Enable interactive widgets in Google Colab
output.enable_custom_widget_manager()

# Display all four widgets
display(widgets.VBox([
    plant_dropdown,
    product_dropdown,
    quality_dropdown,
    units_slider
]))

In [ ]:
from IPython.display import display

def filter_data(plant, product, quality, units_range):
    # Create a copy of the dataframe to avoid modifying the original
    dff = df.copy()

    # 1. Filter by Plant
    if plant != "__ALL__":
        dff = dff[dff["plant_id"].astype(str) == plant]

    # 2. Filter by Product Family
    if product != "__ALL__":
        dff = dff[dff["product_family"].astype(str) == product]

    # 3. Filter by Quality Status
    if quality != "__ALL__":
        dff = dff[dff["quality_status"].astype(str) == quality]

    # 4. Filter by Units Produced Slider
    dff["_temp_units"] = pd.to_numeric(dff["units_produced"], errors="coerce")
    dff = dff[
        (dff["_temp_units"] >= units_range[0]) &
        (dff["_temp_units"] <= units_range[1])
    ]
    dff = dff.drop(columns=["_temp_units"])

    # Display summary and filtered data
    print(f"Matching Rows: {len(dff)}")
    display(dff.head(15)) # Shows the first 15 rows of the filtered data

    return dff

In [ ]:
# Link the widgets to the function arguments
out = widgets.interactive_output(
    filter_data,
    {
        "plant": plant_dropdown,
        "product": product_dropdown,
        "quality": quality_dropdown,
        "units_range": units_slider
    }
)

In [ ]:
# Organize widgets into a clean layout
ui = widgets.VBox([
    widgets.HBox([plant_dropdown, product_dropdown]),
    widgets.HBox([quality_dropdown]),
    units_slider,
    out
])

# Display the interactive dashboard
display(ui)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, desc

spark = SparkSession.builder \
    .appName("LazyExecutionDemo") \
    .getOrCreate()

data = [
    (1, "Hyderabad", "Laptop", 2, 55000),
    (2, "Bangalore", "Mobile", 5, 20000),
    (3, "Chennai", "Tablet", 3, 15000),
    (4, "Hyderabad", "Mobile", 4, 22000),
    (5, "Mumbai", "Laptop", 1, 60000),
    (6, "Bangalore", "Tablet", 6, 12000),
    (7, "Chennai", "Laptop", 2, 58000),
    (8, "Mumbai", "Mobile", 8, 18000),
    (9, "Hyderabad", "Tablet", 5, 14000),
    (10, "Bangalore", "Laptop", 3, 52000),
    (11, "Mumbai", "Tablet", 7, 13000),
    (12, "Chennai", "Mobile", 6, 21000)
]

columns = ["sale_id", "city", "product", "quantity", "price"]

df = spark.createDataFrame(data, columns)

df.show()

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      2|Bangalore| Mobile|       5|20000|
|      3|  Chennai| Tablet|       3|15000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      6|Bangalore| Tablet|       6|12000|
|      7|  Chennai| Laptop|       2|58000|
|      8|   Mumbai| Mobile|       8|18000|
|      9|Hyderabad| Tablet|       5|14000|
|     10|Bangalore| Laptop|       3|52000|
|     11|   Mumbai| Tablet|       7|13000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+



In [ ]:
#transformation 1
filtered_df=df.filter(col("quantity")>3)
print("Transformation created")

#transformation 2
selected_df=filtered_df.select("city","product","quantity")
print("Another Transformation created")

# action
selected_df.show()

Transformation created
Another Transformation created
+---------+-------+--------+
|     city|product|quantity|
+---------+-------+--------+
|Bangalore| Mobile|       5|
|Hyderabad| Mobile|       4|
|Bangalore| Tablet|       6|
|   Mumbai| Mobile|       8|
|Hyderabad| Tablet|       5|
|   Mumbai| Tablet|       7|
|  Chennai| Mobile|       6|
+---------+-------+--------+



In [ ]:
# transformation 1: filter sales
step1=df.filter(col("quantity")>=3)

# transformation 2:calculate revenue
step2=step1.withColumn("revenue",col("quantity")*col("price"))

# transformation 3:groupby city
step3=step2.groupBy("city").agg(sum("revenue").alias("total_revenue"))

# transformation 4:sort results
step4=step3.orderBy(desc("total_revenue"))
print("All transformations defined")

# nothing above reuires spark to execute the query
# action
step4.show()

All transformations defined
+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|       328000|
|   Mumbai|       235000|
|  Chennai|       171000|
|Hyderabad|       158000|
+---------+-------------+



In [ ]:
# define transformation
res=df.filter(col("price")>20000).select("city","product","price").orderBy(desc("price"))
# view spark's execution plan
res.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [price#4L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(price#4L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=178]
      +- Project [city#1, product#2, price#4L]
         +- Filter (isnotnull(price#4L) AND (price#4L > 20000))
            +- Scan ExistingRDD[sale_id#0L,city#1,product#2,quantity#3L,price#4L]




In [ ]:
res.show()

+---------+-------+-----+
|     city|product|price|
+---------+-------+-----+
|   Mumbai| Laptop|60000|
|  Chennai| Laptop|58000|
|Hyderabad| Laptop|55000|
|Bangalore| Laptop|52000|
|Hyderabad| Mobile|22000|
|  Chennai| Mobile|21000|
+---------+-------+-----+



In [ ]:
high_val_sales=df.filter(col("price")>20000)
# action1
high_val_sales.show()
# action2
print("Total records:",high_val_sales.count())

# action3
rec=high_val_sales.collect()
print(rec)

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      7|  Chennai| Laptop|       2|58000|
|     10|Bangalore| Laptop|       3|52000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+

Total records: 6
[Row(sale_id=1, city='Hyderabad', product='Laptop', quantity=2, price=55000), Row(sale_id=4, city='Hyderabad', product='Mobile', quantity=4, price=22000), Row(sale_id=5, city='Mumbai', product='Laptop', quantity=1, price=60000), Row(sale_id=7, city='Chennai', product='Laptop', quantity=2, price=58000), Row(sale_id=10, city='Bangalore', product='Laptop', quantity=3, price=52000), Row(sale_id=12, city='Chennai', product='Mobile', quantity=6, price=21000)]


Practice

In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))

# Read using Spark instead of pandas
sales_df = spark.read.csv(file_name, header=True, inferSchema=True)

print("Dataset loaded successfully as PySpark DataFrame!")
print("Number of records:", sales_df.count())
sales_df.show(5)

Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000 (3).csv
Dataset loaded successfully as PySpark DataFrame!
Number of records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|

In [ ]:
filtered_df = sales_df.filter(col("quantity") > 5)
filtered_df.show(5)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
+--------+----------+--------+----------+--------+----------+---

In [ ]:
sales_df.select("order_id", "product_id", "quantity", "unit_price").show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000001|      P002|       3|      2999|
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000005|      P007|       2|     79999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000009|      P007|       5|     79999|
| O000010|      P006|       5|     55999|
| O000011|      P007|       1|     79999|
| O000012|      P010|       9|      3999|
| O000013|      P013|       6|      5999|
| O000014|      P005|       8|      2499|
| O000015|      P007|       7|     79999|
| O000016|      P006|       1|     55999|
| O000017|      P004|       9|     16999|
| O000018|      P012|       7|      6999|
| O000019|      P014|       5|     29999|
| O000020|      P010|       7|      3999|
+--------+----------+--------+----

In [ ]:
high_price_count = sales_df.filter(col("unit_price") > 20000).count()
print(f"Matching records: {high_price_count}")

Matching records: 6560


In [ ]:
gross_df = sales_df.withColumn("gross_amount", col("quantity") * col("unit_price"))
gross_df.explain(True)  # Inspects the logical and physical execution plans
gross_df.show(5)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(gross_amount, '`*`('quantity, 'unit_price), None)]
+- Relation [order_id#121,order_date#122,store_id#123,product_id#124,quantity#125,unit_price#126,discount_pct#127,payment_method#128,order_status#129,sales_channel#130] csv

== Analyzed Logical Plan ==
order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string, gross_amount: int
Project [order_id#121, order_date#122, store_id#123, product_id#124, quantity#125, unit_price#126, discount_pct#127, payment_method#128, order_status#129, sales_channel#130, (quantity#125 * unit_price#126) AS gross_amount#260]
+- Relation [order_id#121,order_date#122,store_id#123,product_id#124,quantity#125,unit_price#126,discount_pct#127,payment_method#128,order_status#129,sales_channel#130] csv

== Optimized Logical Plan ==
Project [order_id#121, order_date#122, stor

In [ ]:
sales_df.filter(col("order_status") == "Completed") \
    .orderBy(col("unit_price").desc()) \
    .take(10)

[Row(order_id='O000085', order_date=datetime.date(2026, 4, 22), store_id='S03', product_id='P007', quantity=4, unit_price=79999, discount_pct=0, payment_method='Card', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000290', order_date=datetime.date(2026, 6, 26), store_id='S06', product_id='P007', quantity=3, unit_price=79999, discount_pct=10, payment_method='Net Banking', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000098', order_date=datetime.date(2026, 4, 15), store_id='S05', product_id='P007', quantity=1, unit_price=79999, discount_pct=5, payment_method='Net Banking', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000005', order_date=datetime.date(2026, 3, 14), store_id='S06', product_id='P007', quantity=2, unit_price=79999, discount_pct=0, payment_method='Cash', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000107', order_date=datetime.date(2026, 5, 24), store_id='S05', product_id='P007', quantity=9

part B-Chained Transformations

In [ ]:
pipeline_df = (sales_df
    .filter(col("order_status") == "Completed")
    .withColumn("gross_amount", col("quantity") * col("unit_price"))
    .select("order_id", "order_date", "product_id", "quantity", "gross_amount")
    .orderBy(col("gross_amount").desc())
)
pipeline_df.show(10)

+--------+----------+----------+--------+------------+
|order_id|order_date|product_id|quantity|gross_amount|
+--------+----------+----------+--------+------------+
| O001170|2026-01-24|      P007|       9|      719991|
| O002603|2026-03-27|      P007|       9|      719991|
| O001201|2026-02-11|      P007|       9|      719991|
| O000107|2026-05-24|      P007|       9|      719991|
| O001209|2026-04-10|      P007|       9|      719991|
| O000456|2026-02-09|      P007|       9|      719991|
| O001300|2026-04-15|      P007|       9|      719991|
| O000615|2026-04-16|      P007|       9|      719991|
| O001595|2026-06-13|      P007|       9|      719991|
| O000737|2026-05-21|      P007|       9|      719991|
+--------+----------+----------+--------+------------+
only showing top 10 rows


In [ ]:
discounted_df = (sales_df
    .filter(col("discount_pct") >= 10)
    .withColumn("discounted_amount", col("quantity") * col("unit_price") * (1 - col("discount_pct") / 100))
    .orderBy(col("discounted_amount").desc())
)
discounted_df.show(15)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|discounted_amount|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------+
| O006653|2026-06-29|     S01|      P007|       9|     79999|          10|          Card|   Cancelled|       Online|         647991.9|
| O011893|2026-03-26|     S07|      P007|       9|     79999|          10|           UPI|   Completed|       Online|         647991.9|
| O006699|2026-06-25|     S05|      P007|       9|     79999|          10|   Net Banking|   Completed|       Online|         647991.9|
| O003108|2026-02-23|     S08|      P007|       9|     79999|          10|   Net Banking|   Completed|     In Store|         647991.9|
| O007743|2026-05-22|     S08|      P007|       9|     

In [ ]:
sales_df.groupBy("payment_method") \
    .agg(sum("quantity").alias("total_quantity")) \
    .orderBy(desc("total_quantity")) \
    .show()

+--------------+--------------+
|payment_method|total_quantity|
+--------------+--------------+
|          Cash|         31754|
|          Card|         31327|
|           UPI|         30894|
|   Net Banking|         30889|
+--------------+--------------+



In [ ]:
sales_df.groupBy("sales_channel", "order_status") \
    .count() \
    .show()

+-------------+------------+-----+
|sales_channel|order_status|count|
+-------------+------------+-----+
|       Online|   Completed|10797|
|     In Store|   Cancelled|  592|
|       Online|    Returned|  987|
|       Online|   Cancelled|  633|
|     In Store|    Returned| 1013|
|     In Store|   Completed|10978|
+-------------+------------+-----+



In [ ]:
sub_df = sales_df.select("order_id", "store_id", "product_id", "quantity") \
    .drop("store_id")

first_record = sub_df.first()
print(first_record)

Row(order_id='O000001', product_id='P002', quantity=3)


part c joins and aggregations

In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))

# Read using Spark instead of pandas
stores_df = spark.read.csv(file_name, header=True, inferSchema=True)

print("Dataset loaded successfully as PySpark DataFrame!")
print("Number of records:", stores_df.count())
stores_df.show(5)

Saving lazy_execution_stores.csv to lazy_execution_stores (1).csv
Dataset loaded successfully as PySpark DataFrame!
Number of records: 8
+--------+---------+------+-----------+
|store_id|     city|region| store_type|
+--------+---------+------+-----------+
|     S01|Hyderabad| South|       Mall|
|     S02|Bengaluru| South|High Street|
|     S03|  Chennai| South|       Mall|
|     S04|   Mumbai|  West|High Street|
|     S05|     Pune|  West|       Mall|
+--------+---------+------+-----------+
only showing top 5 rows


In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))

# Read using Spark instead of pandas
products_df = spark.read.csv(file_name, header=True, inferSchema=True)

print("Dataset loaded successfully as PySpark DataFrame!")
print("Number of records:", products_df.count())
products_df.show(5)

Saving lazy_execution_products.csv to lazy_execution_products (2).csv
Dataset loaded successfully as PySpark DataFrame!
Number of records: 15
+----------+-------------------+-----------+----------+----------+
|product_id|       product_name|   category|list_price|cost_price|
+----------+-------------------+-----------+----------+----------+
|      P001|     Wireless Mouse|Accessories|       799|       420|
|      P002|Mechanical Keyboard|Accessories|      2999|      1600|
|      P003|          USB-C Hub|Accessories|      1899|       920|
|      P004|    27-inch Monitor|   Displays|     16999|     11400|
|      P005|          Webcam HD|Accessories|      2499|      1400|
+----------+-------------------+-----------+----------+----------+
only showing top 5 rows


In [ ]:
sales_df.join(products_df, "product_id") \
    .select("product_name", "category", "quantity", "unit_price") \
    .show(20)

+-------------------+-----------+--------+----------+
|       product_name|   category|quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|       3|      2999|
|      Smartphone 5G|    Mobiles|       6|     29999|
|     Android Tablet|    Tablets|       6|     22999|
|     Android Tablet|    Tablets|       7|     22999|
|          Laptop 16|  Computers|       2|     79999|
|          Webcam HD|Accessories|       8|      2499|
|  Bluetooth Speaker|      Audio|       8|      3999|
|Mechanical Keyboard|Accessories|       9|      2999|
|          Laptop 16|  Computers|       5|     79999|
|          Laptop 14|  Computers|       5|     55999|
|          Laptop 16|  Computers|       1|     79999|
|  Bluetooth Speaker|      Audio|       9|      3999|
|   External HDD 2TB|    Storage|       6|      5999|
|          Webcam HD|Accessories|       8|      2499|
|          Laptop 16|  Computers|       7|     79999|
|          Laptop 14|  Compu

In [ ]:
sales_df.join(stores_df, "store_id") \
    .groupBy("city") \
    .agg(sum("quantity").alias("total_quantity")) \
    .show()

+---------+--------------+
|     city|total_quantity|
+---------+--------------+
|  Chennai|         15727|
|   Mumbai|         15489|
|  Kolkata|         15619|
|     Pune|         15407|
|    Delhi|         15101|
|Bengaluru|         15718|
|Hyderabad|         15784|
|   Jaipur|         16019|
+---------+--------------+



In [ ]:
(sales_df
    .filter(col("order_status") == "Completed")
    .join(products_df, "product_id")
    .join(stores_df, "store_id")
    .withColumn("revenue", col("quantity") * col("unit_price"))
    .groupBy("category", "region")
    .agg(sum("revenue").alias("total_revenue"))
    .show()
)

+-----------+------+-------------+
|   category|region|total_revenue|
+-----------+------+-------------+
|   Displays| North|     32468090|
|   Displays| South|     45761308|
|    Storage| South|     36976319|
|   Displays|  East|     13616199|
|      Audio|  East|     12043135|
|    Tablets| North|     38270336|
|    Mobiles| South|    120674524|
|    Storage|  East|     12140136|
|    Mobiles| North|     75482543|
|    Mobiles|  East|     41178126|
|Accessories|  East|      9093361|
|      Audio|  West|     25317111|
|    Tablets|  West|     37488370|
|   Displays|  West|     28286336|
|      Audio| South|     34629665|
|      Audio| North|     24327211|
|    Storage|  West|     23673341|
|  Computers| North|    240756425|
|  Computers|  West|    221092729|
|    Mobiles|  West|     80372388|
+-----------+------+-------------+
only showing top 20 rows


In [ ]:
(sales_df
    .filter(col("order_status") == "Completed")
    .join(products_df, "product_id")
    .withColumn("discounted_revenue", col("quantity") * col("unit_price") * (1 - col("discount_pct") / 100))
    .groupBy("product_id", "product_name")
    .agg(sum("discounted_revenue").alias("total_discounted_revenue"))
    .orderBy(desc("total_discounted_revenue"))
    .show(5)
)

+----------+---------------+------------------------+
|product_id|   product_name|total_discounted_revenue|
+----------+---------------+------------------------+
|      P007|      Laptop 16|    5.2272546585000014E8|
|      P006|      Laptop 14|     3.830051604999999E8|
|      P014|  Smartphone 5G|     2.060406317500002E8|
|      P008| Android Tablet|    1.5040656029999995E8|
|      P004|27-inch Monitor|    1.1406074015000013E8|
+----------+---------------+------------------------+
only showing top 5 rows


part d lazy evaluation

In [ ]:
df1 = sales_df.filter(col("quantity") > 2)
print("After filter:", df1)

df2 = df1.select("order_id", "product_id", "quantity", "unit_price")
print("After select:", df2)

df3 = df2.withColumn("total", col("quantity") * col("unit_price"))
print("After withColumn:", df3)

df4 = df3.groupBy("product_id").agg(sum("total").alias("revenue"))
print("After groupBy:", df4)

df5 = df4.orderBy(desc("revenue"))
print("After orderBy:", df5)

# Trigger execution
df5.show()

After filter: DataFrame[order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string]
After select: DataFrame[order_id: string, product_id: string, quantity: int, unit_price: int]
After withColumn: DataFrame[order_id: string, product_id: string, quantity: int, unit_price: int, total: int]
After groupBy: DataFrame[product_id: string, revenue: bigint]
After orderBy: DataFrame[product_id: string, revenue: bigint]
+----------+---------+
|product_id|  revenue|
+----------+---------+
|      P007|599112511|
|      P006|427944358|
|      P014|231502283|
|      P008|170146602|
|      P004|125197635|
|      P015|107736304|
|      P011| 71227085|
|      P012| 56488929|
|      P013| 48531910|
|      P010| 32287926|
|      P002| 22435519|
|      P005| 20109453|
|      P009| 15406293|
|      P003| 14424804|
|      P001|  6540614|
+----------+---------+



Python print() statements output the object representation (e.g., <DataFrame ...>) immediately without executing any computational work. Spark does not process data until .show() is called at the end.

In [ ]:
filtered_sales = sales_df.filter(col("unit_price") > 15000)
filtered_sales.explain()

# Job 1 triggered
filtered_sales.show(5)

# Job 2 triggered (re-executes the plan unless cached)
print("Count:", filtered_sales.count())

== Physical Plan ==
*(1) Filter (isnotnull(unit_price#559) AND (unit_price#559 > 15000))
+- FileScan csv [order_id#554,order_date#555,store_id#556,product_id#557,quantity#558,unit_price#559,discount_pct#560,payment_method#561,order_status#562,sales_channel#563] Batched: false, DataFilters: [isnotnull(unit_price#559), (unit_price#559 > 15000)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000 (3).csv], PartitionFilters: [], PushedFilters: [IsNotNull(unit_price), GreaterThan(unit_price,15000)], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+---------

Observation: Calling show() and count() separately triggers two distinct Spark jobs in the execution DAG because each is a separate action.

In [ ]:
category_totals = sales_df.join(products_df, "product_id") \
    .groupBy("category") \
    .agg(sum("quantity").alias("total_qty"))

results = category_totals.collect()
for row in results:
    print(row)

Row(category='Computers', total_qty=16258)
Row(category='Displays', total_qty=7870)
Row(category='Mobiles', total_qty=16572)
Row(category='Accessories', total_qty=41839)
Row(category='Storage', total_qty=17251)
Row(category='Audio', total_qty=17159)
Row(category='Tablets', total_qty=7915)


Why it's safer: collect() pulls all result records from cluster executor nodes back into the driver node's memory. Collecting raw data (all 25,000+ rows) risks causing an Out-Of-Memory (OOM) error on the driver. In contrast, collecting a small aggregated summary (e.g., a few product categories) transmits a tiny payload that fits safely into memory.

In [ ]:
(sales_df
    .filter(col("order_status") == "Completed")
    .withColumn("discounted_revenue", col("quantity") * col("unit_price") * (1 - col("discount_pct") / 100))
    .write.mode("overwrite")
    .parquet("completed_discounted_revenue.parquet")
)

Trigger Identification: The operation that triggers execution is .parquet(...) (part of the DataFrame write API), which acts as an output action forcing Spark to evaluate the lazy transformation lineage and write out the results.

afternoon hands on

In [1]:
from google.colab import files
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

# Browse and select multiple CSV files
uploaded = files.upload()

# Load selected CSV files into PySpark DataFrames
dataframes = {}

for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df

    print(f"\nFile: {filename}")
    print("Records:", df.count())
    df.show(5)

Saving lazy_execution_stores.csv to lazy_execution_stores.csv
Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv
Saving lazy_execution_products.csv to lazy_execution_products.csv

File: lazy_execution_stores.csv
Records: 8
+--------+---------+------+-----------+
|store_id|     city|region| store_type|
+--------+---------+------+-----------+
|     S01|Hyderabad| South|       Mall|
|     S02|Bengaluru| South|High Street|
|     S03|  Chennai| South|       Mall|
|     S04|   Mumbai|  West|High Street|
|     S05|     Pune|  West|       Mall|
+--------+---------+------+-----------+
only showing top 5 rows

File: lazy_execution_sales_25000.csv
Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+-----------

In [4]:
from pyspark.sql import functions as F
from pyspark import StorageLevel
import time

# Find the sales dataset from the uploaded files
sales_df = None

for filename, df in dataframes.items():
    if {"order_id", "quantity", "unit_price", "order_status"}.issubset(df.columns):
        sales_df = df
        print("Sales file selected:", filename)
        break

if sales_df is None:
    raise ValueError("Sales CSV not found. Please upload the sales dataset.")

print("Total Sales Records:", sales_df.count())
sales_df.show(5)

Sales file selected: lazy_execution_sales_25000.csv
Total Sales Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|

In [6]:
# transformation
completed_df=sales_df.filter(F.col("order_status")=="Completed")
print("Transformation created")
# action 1
print("Completed orders:",completed_df.count())

# action 2
completed_df.show(10)

Transformation created
Completed orders: 21775
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04

In [7]:
# cache the filtered dataframe
completed_df.cache()

print("Is cached:",completed_df.is_cached)

# first action populates cache
print("Cached records:",completed_df.count())

# reuse cached data
completed_df.show(10)

# anothe action using cached data
completed_df.groupBy("product_id").count().show()

Is cached: True
Cached records: 21775
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04-05|     

persist and unpersist

In [8]:
# create filtered dataframe
high_qnt_df=sales_df.filter(F.col("quantity")>5)

# persist using MEMORY AND DISK
high_qnt_df.persist(StorageLevel.MEMORY_AND_DISK)

# trigger execution
high_qnt_df.count()

# reuse persisted data
high_qnt_df.show(10)

# check storage level
print("Storage Level:",high_qnt_df.storageLevel)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|   

In [9]:
# check cache status
print("Before:", completed_df.is_cached)

# remove cache
completed_df.unpersist(blocking=True)

# check again
print("After:", completed_df.is_cached)

# remove the other persisted dataframe
high_qnt_df.unpersist(blocking=True)

Before: True
After: False


DataFrame[order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string]

In [10]:
# Generate 40 batch IDs (0 to 39)
batch_df = spark.range(40).select(
    F.col("id").alias("batch_id")
)

batch_df.show(5)

# Combine each sales record with all 40 batch IDs
large_df = sales_df.crossJoin(batch_df)

print("Original Records:", sales_df.count())
print("Large Dataset Records:", large_df.count())

large_df.show(5)


+--------+
|batch_id|
+--------+
|       0|
|       1|
|       2|
|       3|
|       4|
+--------+
only showing top 5 rows
Original Records: 25000
Large Dataset Records: 1000000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|batch_id|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       0|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       1|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       2|
| O000001|2026-06-22|     S04|      P002|       3|      2999|     

In [11]:
# Create transformations
result_df = large_df.filter(
    F.col("quantity") > 3
).withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy("product_id").agg(
    F.sum("revenue").alias("total_revenue")
)

# First execution
start = time.perf_counter()

result_df.collect()

time1 = time.perf_counter() - start

# Second execution
start = time.perf_counter()

result_df.collect()

time2 = time.perf_counter() - start

print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")

First Execution: 3.56 seconds
Second Execution: 0.202 seconds


PARTITIONING

In [12]:
# check total records
print("Total records: ",sales_df.count())

#check number of partitions
print("Number of partitions:",sales_df.rdd.getNumPartitions())


Total records:  25000
Number of partitions: 1


In [14]:
# create 8 partitions
part_df=sales_df.repartition(8)

print("Original partitions : ",sales_df.rdd.getNumPartitions())

# check number of partitions
print("New partitions:",part_df.rdd.getNumPartitions())

part_df.count()

Original partitions :  1
New partitions: 8


25000

In [16]:
from pyspark.sql import functions as F

# Add partition ID to every record
distribution_df = part_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)

# Count records in each partition
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()


+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 3125|
|           1| 3125|
|           2| 3125|
|           3| 3125|
|           4| 3125|
|           5| 3125|
|           6| 3125|
|           7| 3125|
+------------+-----+



In [20]:
# partition data by store id
store_part_df=sales_df.repartition(8,"store_id")
print("Partitions:",store_part_df.rdd.getNumPartitions())

# inspect partition distribution
store_part_df.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id","store_id") \
    .count() \
    .orderBy("partition_id","store_id") \
    .show(30)


Partitions: 8
+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           0|     S01| 3180|
|           1|     S02| 3121|
|           1|     S08| 3122|
|           3|     S06| 3033|
|           4|     S05| 3117|
|           4|     S07| 3209|
|           5|     S03| 3132|
|           5|     S04| 3086|
+------------+--------+-----+



In [21]:
# create 8 partitions
df8=sales_df.repartition(8)

# reduce repartition
df_repartition=df8.repartition(3)

# reduce coalesce
df_coalesce=df8.coalesce(3)

print("Partitions:",df_repartition.rdd.getNumPartitions())
print("Coalesce:",df_coalesce.rdd.getNumPartitions())

# examine execution plan
print("REPARTITION PLAN")
df_repartition.explain()

print("COALESCE PLAN")
df_coalesce.explain()



Partitions: 3
Coalesce: 3
REPARTITION PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1285]
         +- FileScan csv [order_id#64,order_date#65,store_id#66,product_id#67,quantity#68,unit_price#69,discount_pct#70,payment_method#71,order_status#72,sales_channel#73] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1285]
   +- FileScan csv [order_id#64,order_date#65,store_id#66,product_id#67,quantity#68,unit_price#69,discount_pct#70,payment_method#71,order_status#72,sales_channel#73] Batched: false, DataFilters: [], For

Practice

In [22]:
print(df.rdd.getNumPartitions())

1


In [23]:
new_df = df.repartition(12)

In [25]:
from pyspark.sql.functions import spark_partition_id

new_df.groupBy(spark_partition_id()).count().show()

+--------------------+-----+
|SPARK_PARTITION_ID()|count|
+--------------------+-----+
|                   0| 2084|
|                   1| 2083|
|                   2| 2083|
|                   3| 2083|
|                   4| 2083|
|                   5| 2083|
|                   6| 2083|
|                   7| 2083|
|                   8| 2083|
|                   9| 2084|
|                  10| 2084|
|                  11| 2084|
+--------------------+-----+



In [27]:
n_df = new_df.repartition(6, "product_id")
n_df.show()

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|           0|          Card|   Completed|     In Store|
| O000017|2026-03-19|     S05|      P004|       9|     16999|           0|          Cash|   Completed|       Online|
| O000018|2026-05-15|     S07|      P012|       7|      6999|           5|          Cash|   Completed|       Online|
| O000019|2026-06-21|     S03|      P014|       5|     29999|   

In [31]:
c_df = new_df.coalesce(4)
c_df.rdd.getNumPartitions()

4

In [32]:
print("COALESCE PLAN")
c_df.explain()


COALESCE PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 4
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=1496]
            +- FileScan csv [order_id#64,order_date#65,store_id#66,product_id#67,quantity#68,unit_price#69,discount_pct#70,payment_method#71,order_status#72,sales_channel#73] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...
+- == Initial Plan ==
   Coalesce 4
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=1496]
      +- FileScan csv [order_id#64,order_date#65,store_id#66,product_id#67,quantity#68,unit_price#69,discount_pct#70,payment_method#71,order_status#72,sales_channel#73] Batched: false, D